# Research template

Start every study from here: load a point-in-time daily panel, build split-adjusted returns, define a universe, and evaluate a signal cross-sectionally.

In [ ]:
import datetime as dt
import numpy as np
import polars as pl

from collective_alpha.storage.catalog import Catalog

cat = Catalog()
start, end = dt.date(2023, 1, 1), dt.date.today()
panel = cat.day_panel(start, end)
panel.shape

## Split adjustment

Stored prices are **unadjusted**. Build a cumulative adjustment factor from the splits table so that prices before an execution date are divided by the split ratio.

In [ ]:
splits = cat.sql('select ticker, execution_date, split_from, split_to from splits').pl()
splits = splits.with_columns(ratio=pl.col('split_to') / pl.col('split_from')).sort(['ticker', 'execution_date'])

# cumulative factor to apply to dates strictly before each execution date
adj = (
    panel.select('ticker', 'date')
    .join(splits, on='ticker', how='left')
    .filter(pl.col('date') < pl.col('execution_date'))
    .group_by(['ticker', 'date']).agg(pl.col('ratio').product().alias('adj_factor'))
)
panel = panel.join(adj, on=['ticker', 'date'], how='left').with_columns(pl.col('adj_factor').fill_null(1.0))
panel = panel.with_columns(adj_close=pl.col('close') / pl.col('adj_factor'))
panel.filter(pl.col('ticker') == 'NVDA').select('date', 'close', 'adj_factor', 'adj_close').tail(3)

## Universe

Simple liquidity screen; refine with `ticker_details` (market cap, primary exchange) once synced.

In [ ]:
panel = panel.sort(['ticker', 'date']).with_columns(
    dollar_vol=pl.col('close') * pl.col('volume'),
    ret=pl.col('adj_close') / pl.col('adj_close').shift(1).over('ticker') - 1,
)
panel = panel.with_columns(adv20=pl.col('dollar_vol').rolling_mean(20).over('ticker'))
universe = panel.filter((pl.col('adv20') > 5e6) & (pl.col('close') > 5))
universe.group_by('date').len().tail(3)

## Example signal: 20-day reversal, evaluated by daily rank IC

In [ ]:
sig = universe.with_columns(
    signal=-(pl.col('adj_close') / pl.col('adj_close').shift(20).over('ticker') - 1),
    fwd_ret=pl.col('adj_close').shift(-5).over('ticker') / pl.col('adj_close') - 1,
).drop_nulls(['signal', 'fwd_ret'])

ic = (
    sig.group_by('date')
    .agg(pl.corr(pl.col('signal').rank(), pl.col('fwd_ret').rank()).alias('ic'))
    .sort('date')
)
print('mean IC', ic['ic'].mean(), ' IR', ic['ic'].mean() / ic['ic'].std())
ic.to_pandas().set_index('date')['ic'].rolling(60).mean().plot(figsize=(12, 3), title='60d rolling rank IC');